1. Instalasi Library Llama-CPP Server Berbasis Python (mendukung GPU CUDA yang compatible dengan Google Colab)

In [1]:
# Install abetlen llama-cpp-python CUDA wheel
# Colab sering memakai CUDA 12 user-space walau driver nvidia-smi menampilkan CUDA 13-capable.
# Jika runtime Anda menyediakan libcudart.so.13, ganti cu125 menjadi cu130.
!apt-get -qq update
!apt-get -qq install -y wget curl >/dev/null
!pip install -q --no-cache-dir --upgrade --force-reinstall   requests "huggingface-hub>=0.23.0"   llama-cpp-python   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125

import re
import shutil
import subprocess
import sys

if shutil.which('nvidia-smi') is None:
    raise RuntimeError('No NVIDIA GPU detected. In Colab, choose Runtime > Change runtime type > GPU, then rerun.')

nvidia_smi = subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True)
print(nvidia_smi.stdout)

cuda_version_match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', nvidia_smi.stdout)
cuda_major_minor = tuple(map(int, cuda_version_match.groups())) if cuda_version_match else None
supported_wheels = {
    (11, 8): 'cu118',
    (12, 1): 'cu121',
    (12, 2): 'cu122',
    (12, 3): 'cu123',
    (12, 4): 'cu124',
    (12, 5): 'cu125',
    (13, 0): 'cu130',
    (13, 2): 'cu132',
}

# Colab commonly reports CUDA 12.x. Use the nearest supported wheel that is not newer than the driver-reported CUDA level.
if cuda_major_minor in supported_wheels:
    cuda_wheel = supported_wheels[cuda_major_minor]
elif cuda_major_minor and cuda_major_minor[0] == 12:
    cuda_wheel = 'cu125' if cuda_major_minor[1] >= 5 else 'cu124'
elif cuda_major_minor and cuda_major_minor[0] == 13:
    cuda_wheel = 'cu132' if cuda_major_minor[1] >= 2 else 'cu130'
else:
    cuda_wheel = 'cu124'

extra_index_url = f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_wheel}'
print('Installing llama-cpp-python CUDA wheel from:', extra_index_url)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    'llama-cpp-python[server]', '--extra-index-url', extra_index_url,
], check=True)

import llama_cpp
print('llama-cpp-python version:', getattr(llama_cpp, '__version__', 'unknown'))

# Install Cloudflare Tunnel client (cloudflared)
!wget -q -O /tmp/cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i /tmp/cloudflared.deb >/dev/null 2>&1 || apt-get -f install -y
!cloudflared --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 250.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 161.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 209.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 207.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 181.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 277.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.3/224.3 kB 282.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 256.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 216.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 226.7 MB/s eta 0:0

In [2]:
# 6.1) Load Gemma4-12B GGUF + mmproj memakai Python wrapper
from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

WRAPPER_MODEL_REPO = 'ggml-org/gemma-4-12B-it-GGUF'
WRAPPER_MODEL_FILE = 'gemma-4-12B-it-Q4_K_M.gguf'
WRAPPER_MMPROJ_FILE = 'mmproj-gemma-4-12B-it-Q8_0.gguf'

wrapper_chat_handler = Gemma4ChatHandler.from_pretrained(
    repo_id=WRAPPER_MODEL_REPO,
    filename=WRAPPER_MMPROJ_FILE,
    verbose=False,
)

wrapper_llm = Llama.from_pretrained(
    repo_id=WRAPPER_MODEL_REPO,
    filename=WRAPPER_MODEL_FILE,
    chat_handler=wrapper_chat_handler,
    n_gpu_layers=-1,
    n_ctx=8192,
    flash_attn=True,
    verbose=False,
)

print('Loaded wrapper model:', WRAPPER_MODEL_REPO)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_kv_cache_iswa: using full-size SWA cache (ref: https://github.com/ggml-org/llama.cpp/pull/13194#issuecomment-2868343055)


Loaded wrapper model: ggml-org/gemma-4-12B-it-GGUF


Aplikasi Web untuk Membuat Mermaid Flowchart dengan LLM dan Mermaid v11

In [3]:
# Single-cell Mermaid flowchart generator + Mermaid v11 renderer + Cloudflare Tunnel
# Requirement: wrapper_llm must already exist in this notebook.

!pip -q install flask werkzeug requests

import os
import re
import signal
import shutil
import subprocess
import threading
import time
import uuid

import requests
from flask import Flask, jsonify, render_template_string, request
from werkzeug.serving import make_server


# ============================================================
# Configuration
# ============================================================

APP_PORT = 7860
MERMAID_VERSION = "11"
CLOUDFLARED_LOG_PATH = "/content/colab-mermaid-flowchart-cloudflared.log"

DEFAULT_FLOWCHART_REQUEST = (
    "Create a clear Mermaid flowchart for an online order process. "
    "Include product selection, checkout, payment validation, inventory check, "
    "packing, shipping, delivery confirmation, and failure paths."
)

MERMAID_GENERATION_INSTRUCTIONS = """
You generate Mermaid version 11 flowchart code.
Return only Mermaid code, with no Markdown fences and no explanation.
Use flowchart TD or flowchart LR unless the user requests another direction.
Use short, readable node labels wrapped in quotes when labels contain spaces.
Use stable node ids made from letters, numbers, and underscores only.
Use Mermaid flowchart syntax supported by Mermaid 11.
Keep the diagram readable by grouping related steps with subgraph when useful.
Do not include raw HTML, JavaScript, external URLs, or styling that requires plugins.
""".strip()


# ============================================================
# Global job storage
# ============================================================

jobs = {}
jobs_lock = threading.Lock()

# Many local LLM wrappers are not fully thread-safe.
# This lock ensures only one wrapper_llm call runs at a time.
llm_lock = threading.Lock()


# ============================================================
# Helper functions
# ============================================================

def update_job(job_id, **kwargs):
    with jobs_lock:
        if job_id in jobs:
            jobs[job_id].update(kwargs)


def get_job_copy(job_id):
    with jobs_lock:
        job = jobs.get(job_id)
        if job is None:
            return None
        return dict(job)


def extract_response_text(response):
    try:
        return response["choices"][0]["message"]["content"]
    except Exception:
        return str(response)


def extract_mermaid_code(text):
    text = (text or "").strip()

    fence_match = re.search(
        r"```(?:mermaid|mmd)?\s*([\s\S]*?)```",
        text,
        flags=re.IGNORECASE,
    )
    if fence_match:
        text = fence_match.group(1).strip()

    lines = text.splitlines()

    while lines and lines[0].strip().lower() in {"mermaid", "mmd"}:
        lines.pop(0)

    start_index = None
    flowchart_start = re.compile(
        r"^\s*(flowchart|graph)\s+(TD|TB|BT|RL|LR)\b",
        flags=re.IGNORECASE,
    )

    for index, line in enumerate(lines):
        if flowchart_start.search(line):
            start_index = index
            break

    if start_index is not None:
        lines = lines[start_index:]

    trailing_prose = re.compile(
        r"^\s*(explanation|note|notes|summary)\s*:",
        flags=re.IGNORECASE,
    )
    for index, line in enumerate(lines):
        if trailing_prose.search(line):
            lines = lines[:index]
            break

    return "\n".join(lines).strip()


def looks_like_flowchart(code):
    return bool(
        re.match(
            r"^\s*(flowchart|graph)\s+(TD|TB|BT|RL|LR)\b",
            code or "",
            flags=re.IGNORECASE,
        )
    )


def generate_mermaid_with_llm(flowchart_request):
    """
    Generate Mermaid flowchart code using the existing wrapper_llm object.

    Important:
    The Flask server runs in the same notebook Python process,
    so it can access wrapper_llm directly.
    """

    if "wrapper_llm" not in globals():
        raise RuntimeError(
            "wrapper_llm is not defined. Run the model-loading cell before this app cell."
        )

    user_prompt = f"""{MERMAID_GENERATION_INSTRUCTIONS}

User flowchart request:
{flowchart_request.strip()}

Return only valid Mermaid v11 flowchart code."""

    with llm_lock:
        response = wrapper_llm.create_chat_completion(
            messages=[
                {
                    "role": "user",
                    "content": user_prompt,
                }
            ],
            max_tokens=2048,
            temperature=0.1,
        )

    raw_text = extract_response_text(response)
    mermaid_code = extract_mermaid_code(raw_text)

    return raw_text, mermaid_code


def process_generation_job(job_id, flowchart_request):
    """
    Runs in a background thread.

    Long LLM processing is detached from the browser request,
    so Cloudflare or browser request timeouts do not stop the backend job.
    """

    try:
        update_job(
            job_id,
            status="processing",
            progress=15,
            message="Sending prompt to the model.",
        )

        raw_response, mermaid_code = generate_mermaid_with_llm(flowchart_request)

        update_job(
            job_id,
            status="processing",
            progress=85,
            message="Preparing Mermaid code for rendering.",
            raw_response=raw_response,
            mermaid_code=mermaid_code,
        )

        warning = None
        if not looks_like_flowchart(mermaid_code):
            warning = (
                "The model response was cleaned, but it does not start with "
                "a Mermaid flowchart declaration. You can edit the code and render again."
            )

        update_job(
            job_id,
            status="done",
            progress=100,
            message="Generated Mermaid flowchart code.",
            raw_response=raw_response,
            mermaid_code=mermaid_code,
            warning=warning,
        )

    except Exception as exc:
        update_job(
            job_id,
            status="error",
            progress=100,
            message=f"Generation failed: {exc}",
            error=str(exc),
        )


def stop_process_on_port(port):
    """
    Stop old external process using this port.
    Useful in Colab when a previous Flask server is still running.
    """

    try:
        if shutil.which("bash") is None:
            return

        result = subprocess.run(
            ["bash", "-lc", f"fuser {port}/tcp 2>/dev/null || true"],
            capture_output=True,
            text=True,
        )

        pid_text = result.stdout.strip()

        if not pid_text:
            return

        current_pid = os.getpid()

        for item in pid_text.split():
            try:
                pid = int(item)

                if pid == current_pid:
                    continue

                print(f"Stopping old process {pid} using port {port}...")
                os.kill(pid, signal.SIGTERM)

            except Exception:
                pass

        time.sleep(1)

    except Exception as exc:
        print(f"Warning: could not check/stop old process on port {port}: {exc}")


def install_cloudflared_if_needed():
    if shutil.which("cloudflared") is not None:
        return

    print("Installing cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            "/tmp/cloudflared.deb",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        ],
        check=True,
    )

    result = subprocess.run(
        ["dpkg", "-i", "/tmp/cloudflared.deb"],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        subprocess.run(
            ["apt-get", "-f", "install", "-y"],
            check=True,
        )


def stop_previous_cloudflared_for_port(port):
    """
    Stop old cloudflared process created for this Mermaid app.
    """

    try:
        if mermaid_cloudflare_proc.poll() is None:
            print("Stopping previous Cloudflare Tunnel process...")
            mermaid_cloudflare_proc.terminate()
            mermaid_cloudflare_proc.wait(timeout=10)
    except Exception:
        pass

    try:
        if shutil.which("bash") is None:
            return

        pattern = f"cloudflared.*(127\\.0\\.0\\.1|localhost):{port}"
        result = subprocess.run(
            ["bash", "-lc", f"pgrep -f '{pattern}' || true"],
            capture_output=True,
            text=True,
        )

        current_pid = os.getpid()

        for item in result.stdout.strip().split():
            try:
                pid = int(item)

                if pid == current_pid:
                    continue

                print(f"Stopping old cloudflared process {pid} for port {port}...")
                os.kill(pid, signal.SIGTERM)

            except Exception:
                pass

        time.sleep(1)

    except Exception as exc:
        print(f"Warning: could not stop old cloudflared process: {exc}")


def start_cloudflare_tunnel(port):
    """
    Start Cloudflare Tunnel and print public URL.
    """

    install_cloudflared_if_needed()
    stop_previous_cloudflared_for_port(port)

    cf_cmd = [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{port}",
        "--no-autoupdate",
    ]

    log_file = open(
        CLOUDFLARED_LOG_PATH,
        "w",
        encoding="utf-8",
        buffering=1,
    )

    proc = subprocess.Popen(
        cf_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    public_url_box = {"url": None}
    url_ready = threading.Event()

    def read_cloudflared_output():
        for line in proc.stdout:
            line = line.rstrip()
            log_file.write(line + "\n")
            print("[cloudflared]", line)

            match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)

            if match and public_url_box["url"] is None:
                public_url_box["url"] = match.group(0)
                url_ready.set()

    reader_thread = threading.Thread(
        target=read_cloudflared_output,
        daemon=True,
    )
    reader_thread.start()

    if url_ready.wait(timeout=60):
        print()
        print("=" * 70)
        print("Mermaid flowchart app public URL:")
        print(public_url_box["url"])
        print("=" * 70)
        print()
    else:
        print()
        print("No Cloudflare public URL detected yet.")
        print("Cloudflared log:", CLOUDFLARED_LOG_PATH)

    print("Cloudflared PID:", proc.pid)

    return proc, public_url_box["url"]


# ============================================================
# Flask app
# ============================================================

app = Flask(__name__)

PAGE = r"""
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Mermaid Flowchart Generator</title>

  <style>
    :root {
      color-scheme: light dark;
      font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
      background: #f6f8fb;
      color: #172033;
    }

    * { box-sizing: border-box; }

    body {
      margin: 0;
      min-height: 100vh;
      background: #f6f8fb;
      color: #172033;
    }

    button, textarea { font: inherit; }

    button {
      border: 0;
      border-radius: 6px;
      padding: 10px 14px;
      background: #0f766e;
      color: #ffffff;
      font-weight: 700;
      cursor: pointer;
      min-height: 42px;
    }

    button:hover { background: #0b5f59; }
    button:disabled { background: #94a3b8; cursor: not-allowed; }

    .secondary-button {
      border: 1px solid #b8c3d4;
      background: #ffffff;
      color: #172033;
    }

    .secondary-button:hover { background: #eef3f8; }

    .shell {
      width: min(1480px, calc(100vw - 28px));
      margin: 0 auto;
      padding: 18px 0 24px;
    }

    .topbar {
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 16px;
      margin-bottom: 16px;
    }

    h1, h2 {
      margin: 0;
      letter-spacing: 0;
    }

    h1 {
      font-size: clamp(22px, 3vw, 34px);
      line-height: 1.08;
    }

    h2 {
      font-size: 15px;
      line-height: 1.2;
    }

    .title-row {
      display: flex;
      align-items: center;
      flex-wrap: wrap;
      gap: 10px;
    }

    .badge, .runtime {
      display: inline-flex;
      align-items: center;
      min-height: 30px;
      border-radius: 999px;
      padding: 5px 10px;
      border: 1px solid #cfd8e6;
      background: #ffffff;
      color: #475569;
      font-size: 13px;
      font-weight: 700;
      white-space: nowrap;
    }

    .runtime[data-state="processing"] {
      border-color: #f59e0b;
      color: #8a4b00;
      background: #fff7e6;
    }

    .runtime[data-state="done"] {
      border-color: #14b8a6;
      color: #0f766e;
      background: #e7fbf8;
    }

    .runtime[data-state="error"] {
      border-color: #ef4444;
      color: #b91c1c;
      background: #fff0f0;
    }

    .layout {
      display: grid;
      grid-template-columns: minmax(280px, 0.8fr) minmax(320px, 1fr) minmax(420px, 1.4fr);
      gap: 14px;
      align-items: stretch;
    }

    .panel, .status-band {
      border: 1px solid #d8e0ea;
      border-radius: 8px;
      background: #ffffff;
    }

    .panel {
      min-height: 540px;
      display: flex;
      flex-direction: column;
      overflow: hidden;
    }

    .panel-head {
      min-height: 56px;
      padding: 12px 14px;
      border-bottom: 1px solid #d8e0ea;
      display: flex;
      align-items: center;
      justify-content: space-between;
      gap: 10px;
      background: #fbfcfe;
    }

    .panel-actions {
      display: flex;
      align-items: center;
      gap: 8px;
      flex-wrap: wrap;
      justify-content: flex-end;
    }

    form, .code-wrap, .preview-wrap {
      flex: 1;
      display: flex;
      flex-direction: column;
      min-height: 0;
    }

    .field-wrap, .code-wrap { padding: 14px; }

    textarea {
      width: 100%;
      flex: 1;
      min-height: 320px;
      resize: vertical;
      border: 1px solid #c7d1df;
      border-radius: 6px;
      padding: 12px;
      background: #fbfcfe;
      color: #172033;
      line-height: 1.45;
    }

    #codeEditor {
      font-family: "JetBrains Mono", "Cascadia Code", Consolas, monospace;
      font-size: 13px;
      resize: none;
      min-height: 0;
    }

    .form-actions {
      padding: 0 14px 14px;
      display: flex;
      gap: 10px;
      justify-content: flex-end;
    }

    .preview-wrap {
      padding: 14px;
      background: #f8fafc;
    }

    .diagram {
      flex: 1;
      min-height: 420px;
      border: 1px solid #cfd8e6;
      border-radius: 6px;
      background: #ffffff;
      display: grid;
      place-items: center;
      overflow: auto;
      padding: 18px;
    }

    .diagram svg {
      max-width: 100%;
      height: auto;
    }

    .placeholder {
      color: #64748b;
      text-align: center;
      max-width: 32ch;
      line-height: 1.45;
    }

    .error-box {
      margin-top: 10px;
      border: 1px solid #fca5a5;
      border-radius: 6px;
      background: #fff1f2;
      color: #991b1b;
      padding: 10px;
      white-space: pre-wrap;
      line-height: 1.4;
    }

    .error-box[hidden] { display: none; }

    .status-band {
      margin-top: 14px;
      padding: 12px 14px;
    }

    .progress-track {
      height: 12px;
      width: 100%;
      overflow: hidden;
      border-radius: 999px;
      background: #e2e8f0;
      border: 1px solid #ccd6e3;
      margin-bottom: 10px;
    }

    .progress-bar {
      width: 0%;
      height: 100%;
      border-radius: inherit;
      background: #1d4ed8;
      transition: width 0.25s ease;
    }

    .status-text {
      margin: 0;
      min-height: 22px;
      white-space: pre-wrap;
      color: #475569;
      font-size: 13px;
      line-height: 1.4;
      font-family: "JetBrains Mono", "Cascadia Code", Consolas, monospace;
    }

    @media (max-width: 1180px) {
      .layout { grid-template-columns: 1fr 1fr; }
      .preview-panel { grid-column: 1 / -1; }
    }

    @media (max-width: 760px) {
      .shell {
        width: min(100% - 20px, 720px);
        padding-top: 12px;
      }

      .topbar, .layout {
        display: grid;
        grid-template-columns: 1fr;
      }

      .panel { min-height: 420px; }
      .diagram { min-height: 340px; }
      .panel-head, .form-actions { align-items: stretch; }
      .panel-head, .panel-actions, .form-actions { flex-direction: column; }
      button { width: 100%; }
    }

    @media (prefers-color-scheme: dark) {
      :root, body {
        background: #10151d;
        color: #f7fafc;
      }

      .panel, .status-band, .badge, .runtime {
        background: #171e29;
        border-color: #334155;
      }

      .panel-head {
        background: #1c2532;
        border-color: #334155;
      }

      textarea, .diagram {
        background: #111827;
        border-color: #405169;
        color: #f7fafc;
      }

      .preview-wrap { background: #151c27; }

      .secondary-button {
        background: #1f2937;
        border-color: #475569;
        color: #f7fafc;
      }

      .secondary-button:hover { background: #293548; }
      .badge, .runtime, .status-text, .placeholder { color: #cbd5e1; }

      .progress-track {
        background: #253043;
        border-color: #405169;
      }

      .error-box {
        background: #3b161a;
        color: #fecdd3;
        border-color: #7f1d1d;
      }
    }
  </style>
</head>

<body>
  <div class="shell">
    <header class="topbar">
      <div class="title-row">
        <h1>Mermaid Flowchart Generator</h1>
        <span class="badge">Mermaid v{{ mermaid_version }}</span>
      </div>
      <div class="runtime" id="runtimeBadge" data-state="ready">Ready</div>
    </header>

    <main class="layout">
      <section class="panel input-panel">
        <div class="panel-head">
          <h2>Prompt</h2>
        </div>
        <form id="flowchartForm">
          <div class="field-wrap">
            <textarea id="promptBox" name="prompt" spellcheck="true">{{ default_prompt }}</textarea>
          </div>
          <div class="form-actions">
            <button type="submit" id="generateButton">Generate Diagram</button>
          </div>
        </form>
      </section>

      <section class="panel code-panel">
        <div class="panel-head">
          <h2>Mermaid Code</h2>
          <div class="panel-actions">
            <button class="secondary-button" type="button" id="renderButton">Render</button>
          </div>
        </div>
        <div class="code-wrap">
          <textarea id="codeEditor" spellcheck="false"></textarea>
        </div>
      </section>

      <section class="panel preview-panel">
        <div class="panel-head">
          <h2>Preview</h2>
          <div class="panel-actions">
            <button class="secondary-button" type="button" id="downloadSvgButton">SVG</button>
            <button class="secondary-button" type="button" id="downloadPngButton">PNG</button>
          </div>
        </div>
        <div class="preview-wrap">
          <div class="diagram" id="diagramBox">
            <div class="placeholder">Generated flowchart preview appears here.</div>
          </div>
          <div class="error-box" id="renderError" hidden></div>
        </div>
      </section>
    </main>

    <section class="status-band">
      <div class="progress-track" aria-hidden="true">
        <div class="progress-bar" id="progressBar"></div>
      </div>
      <pre class="status-text" id="statusBox">Status: ready.</pre>
    </section>
  </div>

<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@{{ mermaid_version }}/dist/mermaid.esm.min.mjs";

mermaid.initialize({
  startOnLoad: false,
  securityLevel: "strict",
  theme: "base",
  flowchart: {
    htmlLabels: false,
    curve: "basis"
  },
  themeVariables: {
    primaryColor: "#dff7f3",
    primaryTextColor: "#172033",
    primaryBorderColor: "#0f766e",
    lineColor: "#1d4ed8",
    secondaryColor: "#fff7e6",
    tertiaryColor: "#f8fafc"
  }
});

const flowchartForm = document.getElementById("flowchartForm");
const promptBox = document.getElementById("promptBox");
const generateButton = document.getElementById("generateButton");
const renderButton = document.getElementById("renderButton");
const downloadSvgButton = document.getElementById("downloadSvgButton");
const downloadPngButton = document.getElementById("downloadPngButton");
const codeEditor = document.getElementById("codeEditor");
const diagramBox = document.getElementById("diagramBox");
const renderError = document.getElementById("renderError");
const runtimeBadge = document.getElementById("runtimeBadge");
const statusBox = document.getElementById("statusBox");
const progressBar = document.getElementById("progressBar");

let pollTimer = null;
let renderCounter = 0;
let lastRenderedCode = "";

function setRuntime(label, state) {
  runtimeBadge.textContent = label;
  runtimeBadge.dataset.state = state || "ready";
}

function setProgress(percent) {
  const safePercent = Math.max(0, Math.min(100, Number(percent) || 0));
  progressBar.style.width = safePercent + "%";
}

function setStatus(lines) {
  statusBox.textContent = Array.isArray(lines) ? lines.join("\n") : String(lines || "");
}

function showRenderError(message) {
  renderError.hidden = !message;
  renderError.textContent = message || "";
}

async function renderMermaid(code) {
  const trimmed = (code || "").trim();

  if (!trimmed) {
    diagramBox.innerHTML = '<div class="placeholder">Generated flowchart preview appears here.</div>';
    showRenderError("");
    lastRenderedCode = "";
    return;
  }

  try {
    showRenderError("");
    renderCounter += 1;
    const id = "mermaid-flowchart-" + Date.now() + "-" + renderCounter;
    const result = await mermaid.render(id, trimmed);
    diagramBox.innerHTML = result.svg;

    if (typeof result.bindFunctions === "function") {
      result.bindFunctions(diagramBox);
    }

    lastRenderedCode = trimmed;
  } catch (error) {
    showRenderError(error.message || String(error));
  }
}

async function submitPrompt(event) {
  event.preventDefault();

  if (pollTimer !== null) {
    clearInterval(pollTimer);
    pollTimer = null;
  }

  const prompt = promptBox.value.trim();
  if (!prompt) {
    setStatus("Status: prompt is empty.");
    return;
  }

  generateButton.disabled = true;
  setRuntime("Generating", "processing");
  setProgress(5);
  setStatus("Status: sending prompt to backend.");
  showRenderError("");

  try {
    const response = await fetch("/submit_job", {
      method: "POST",
      headers: {
        "Content-Type": "application/json"
      },
      body: JSON.stringify({ prompt })
    });

    const result = await response.json();

    if (!response.ok || !result.success) {
      throw new Error(result.error || "Request failed.");
    }

    setProgress(result.progress || 10);
    setStatus([
      "Job ID: " + result.job_id,
      "Status: queued",
      "Message: " + result.message
    ]);

    startPolling(result.job_id);
  } catch (error) {
    generateButton.disabled = false;
    setRuntime("Error", "error");
    setProgress(100);
    setStatus("Status: " + (error.message || String(error)));
  }
}

function startPolling(jobId) {
  pollTimer = setInterval(async function() {
    try {
      const response = await fetch("/job_status/" + jobId, {
        method: "GET",
        cache: "no-store"
      });

      const result = await response.json();

      if (!response.ok || !result.success) {
        throw new Error(result.error || "Status check failed.");
      }

      setProgress(result.progress);
      setStatus([
        "Job ID: " + jobId,
        "Status: " + result.status,
        "Message: " + result.message,
        result.warning ? "Warning: " + result.warning : ""
      ].filter(Boolean));

      if (result.status === "done") {
        clearInterval(pollTimer);
        pollTimer = null;
        generateButton.disabled = false;
        setRuntime("Rendered", "done");
        codeEditor.value = result.mermaid_code || "";
        await renderMermaid(codeEditor.value);
      }

      if (result.status === "error") {
        clearInterval(pollTimer);
        pollTimer = null;
        generateButton.disabled = false;
        setRuntime("Error", "error");
        showRenderError(result.error || result.message || "Generation failed.");
      }
    } catch (error) {
      setStatus([
        "Job ID: " + jobId,
        "Status: polling error",
        "Message: " + (error.message || String(error))
      ]);
    }
  }, 2000);
}

function downloadBlob(filename, blob) {
  const url = URL.createObjectURL(blob);
  const link = document.createElement("a");
  link.href = url;
  link.download = filename;
  document.body.appendChild(link);
  link.click();
  link.remove();
  URL.revokeObjectURL(url);
}

function currentSvgText() {
  const svg = diagramBox.querySelector("svg");
  if (!svg) {
    throw new Error("No rendered SVG is available.");
  }
  return new XMLSerializer().serializeToString(svg);
}

function downloadSvg() {
  try {
    const svgText = currentSvgText();
    downloadBlob(
      "mermaid-flowchart.svg",
      new Blob([svgText], { type: "image/svg+xml;charset=utf-8" })
    );
  } catch (error) {
    showRenderError(error.message || String(error));
  }
}

function downloadPng() {
  try {
    const svg = diagramBox.querySelector("svg");
    const svgText = currentSvgText();
    const viewBox = svg.viewBox && svg.viewBox.baseVal;
    const bounds = svg.getBoundingClientRect();
    const width = Math.max(800, Math.ceil((viewBox && viewBox.width) || bounds.width || 1200));
    const height = Math.max(500, Math.ceil((viewBox && viewBox.height) || bounds.height || 800));
    const scale = 2;
    const svgBlob = new Blob([svgText], { type: "image/svg+xml;charset=utf-8" });
    const url = URL.createObjectURL(svgBlob);
    const image = new Image();

    image.onload = function() {
      const canvas = document.createElement("canvas");
      canvas.width = width * scale;
      canvas.height = height * scale;
      const context = canvas.getContext("2d");
      context.fillStyle = "#ffffff";
      context.fillRect(0, 0, canvas.width, canvas.height);
      context.drawImage(image, 0, 0, canvas.width, canvas.height);
      URL.revokeObjectURL(url);

      canvas.toBlob(function(blob) {
        if (blob) {
          downloadBlob("mermaid-flowchart.png", blob);
        }
      }, "image/png");
    };

    image.onerror = function() {
      URL.revokeObjectURL(url);
      showRenderError("Could not convert the rendered SVG to PNG.");
    };

    image.src = url;
  } catch (error) {
    showRenderError(error.message || String(error));
  }
}

flowchartForm.addEventListener("submit", submitPrompt);
renderButton.addEventListener("click", function() {
  renderMermaid(codeEditor.value);
});
downloadSvgButton.addEventListener("click", downloadSvg);
downloadPngButton.addEventListener("click", downloadPng);
codeEditor.addEventListener("input", function() {
  if (codeEditor.value.trim() !== lastRenderedCode) {
    setRuntime("Edited", "ready");
  }
});

setProgress(0);
</script>
</body>
</html>
"""


@app.route("/", methods=["GET"])
def index():
    return render_template_string(
        PAGE,
        default_prompt=DEFAULT_FLOWCHART_REQUEST,
        mermaid_version=MERMAID_VERSION,
    )


@app.route("/submit_job", methods=["POST"])
def submit_job():
    payload = request.get_json(silent=True) or {}
    flowchart_request = (
        payload.get("prompt")
        or request.form.get("prompt")
        or DEFAULT_FLOWCHART_REQUEST
    ).strip()

    if not flowchart_request:
        return jsonify(
            {
                "success": False,
                "error": "Prompt is empty.",
            }
        ), 400

    job_id = uuid.uuid4().hex

    with jobs_lock:
        jobs[job_id] = {
            "status": "queued",
            "progress": 10,
            "message": "Prompt received. Job queued.",
            "prompt": flowchart_request,
            "mermaid_code": "",
            "raw_response": "",
            "warning": None,
            "error": None,
            "created_at": time.time(),
        }

    worker = threading.Thread(
        target=process_generation_job,
        args=(job_id, flowchart_request),
        daemon=True,
    )
    worker.start()

    return jsonify(
        {
            "success": True,
            "job_id": job_id,
            "progress": 10,
            "message": "Generation started in the background.",
        }
    )


@app.route("/job_status/<job_id>", methods=["GET"])
def job_status(job_id):
    job = get_job_copy(job_id)

    if job is None:
        return jsonify(
            {
                "success": False,
                "error": "Job not found.",
            }
        ), 404

    return jsonify(
        {
            "success": True,
            "job_id": job_id,
            **job,
        }
    )


@app.route("/health")
def health():
    return jsonify(
        {
            "ok": True,
            "port": APP_PORT,
            "mermaid_version": MERMAID_VERSION,
            "wrapper_llm_available": "wrapper_llm" in globals(),
            "jobs": len(jobs),
        }
    )


# ============================================================
# Run Flask in same notebook process
# ============================================================

class ServerThread(threading.Thread):
    def __init__(self, flask_app, host="0.0.0.0", port=7860):
        super().__init__()
        self.host = host
        self.port = port
        self.server = make_server(host, port, flask_app, threaded=True)
        self.ctx = flask_app.app_context()
        self.ctx.push()
        self.daemon = True

    def run(self):
        self.server.serve_forever()

    def shutdown(self):
        self.server.shutdown()


# Stop previous in-notebook server if this cell was run before.
try:
    mermaid_app_server.shutdown()
    print("Stopped previous in-notebook Mermaid app server.")
except Exception:
    pass

# Stop old external process using the same port, if any.
stop_process_on_port(APP_PORT)

# Start Flask server.
mermaid_app_server = ServerThread(
    app,
    host="0.0.0.0",
    port=APP_PORT,
)

mermaid_app_server.start()

# Wait until local server is ready.
ready = False

for _ in range(30):
    try:
        r = requests.get(f"http://127.0.0.1:{APP_PORT}/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass

    time.sleep(1)

if not ready:
    raise RuntimeError("Mermaid flowchart app did not start.")

print("Mermaid flowchart app running locally:")
print(f"  http://127.0.0.1:{APP_PORT}")
print("This Flask server runs in the same notebook process, so it can access wrapper_llm.")
print()

# Start Cloudflare Tunnel.
mermaid_cloudflare_proc, mermaid_public_url = start_cloudflare_tunnel(APP_PORT)


INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:27] "GET /health HTTP/1.1" 200 -


Mermaid flowchart app running locally:
  http://127.0.0.1:7860
This Flask server runs in the same notebook process, so it can access wrapper_llm.

[cloudflared] 2026-07-13T02:23:29Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
[cloudflared] 2026-07-13T02:23:29Z INF Requesting new quick Tunnel on trycloudflare.com...
[cloudflared] 2026-07-13T02:23:34Z INF +--------------------------------------------------------------------------------------------+
[cloudflared

[cloudflared] 2026-07-13T02:23:34Z INF Generated Connector ID: 1b66e5de-d815-4e00-847b-f432bf7c31ac
[cloudflared] 2026-07-13T02:23:34Z INF Initial protocol quic
[cloudflared] 2026-07-13T02:23:34Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
[cloudflared] 2026-07-13T02:23:34Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
[cloudflared] 2026/07/13 02:23:34 failed to sufficiently increase receive buffer size (was: 208 kiB, wanted: 7168 kiB, got: 416 kiB). See https://github.com/quic-go/quic-go/wiki/UDP-Buffer-Sizes for details.
[cloudflared] 2026-07-13T02:23:34Z INF ICMP proxy will use 172.28.0.12 as source for IPv4
[cloudflared] 2026-07-13T02:23:34Z INF ICMP proxy will use ::1 in zone lo as source for IPv6
[cloudflared] 2026-07-13T02:23:34Z INF Starting metrics server on 127.0.0.1:20241/metrics
[cloudflared] 2026-07-13T02:23:34Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 event=0 ip=198.41.192.77
[cloudflared] 2026-0

INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:46] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:47] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:49] "POST /submit_job HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:52] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:54] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:56] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:23:58] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:24:00] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:24:02] "GET /job_status/06d76b415889483197fa5f2099f3d46c HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Jul/2026 02:29:34] "PO